# Build a web edition from MEI

**Workflow 1 — editing music with the MEI data format.** This notebook is the last step: it turns the result of a facsimile notebook into a static web edition, a folder of files that any web host can serve. Visitors' browsers draw the score with Verovio at its encoded page and system breaks and show the scan beside it, with a box for every bar. Hovering a symbol shows its bar and `xml:id`, and a link such as `#suite/10` opens bar 10.

| Result of | Example | Facsimile |
| --- | --- | --- |
| [`mei_pdf_facsimile_integration.ipynb`](mei_pdf_facsimile_integration.ipynb) (default) | Buxtehude's Suite, 89 bars on four pages | page images of a PDF |
| [`mei_single_file_iiif_integration.ipynb`](mei_single_file_iiif_integration.ipynb) | one page of DdT vol. 11, 17 bars | IIIF images of the Bayerische Staatsbibliothek |
| [`mei_pdf_facsimile_guided.ipynb`](mei_pdf_facsimile_guided.ipynb) | the same Suite, linked in the guided version | page images of a PDF |

Run that notebook first. If its result is missing, this notebook says so and builds a site of the score alone from the example MEI in `test_corpus/`, so that it still runs to the end.

The notebook writes `site.json` and the site folder `_site/` next to the result (gitignored, like all of `converted_mei/`). For the IIIF result it also adds missing page-break links to the MEI (step 2). The scans are under **CC BY-NC-SA 4.0** (Bayerische Staatsbibliothek): publish a site that shows them only as that licence allows.

Requires CAMAT 0.2.8 or newer. See also [From MEI to a web edition](../docs/guides/web-edition.md).

In [ ]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
try:
    from camat import display_path, link_pb_to_surface, resolve_repo_path
except ModuleNotFoundError:
    import setup_camat
    from camat import display_path, link_pb_to_surface, resolve_repo_path

try:
    from camat.web_edition import SiteConfigError, build_site
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "This notebook needs CAMAT 0.2.8 or newer (camat.web_edition). "
        "Upgrade it with  %pip install -U camat  and restart the kernel."
    ) from None

import functools
import http.server
import json
import os
import threading
from pathlib import Path

## 1. Choose the result

Set `RESULT` to the notebook whose result you want to publish, and run the cell again after every change.

In [ ]:
# "pdf"    = mei_pdf_facsimile_integration.ipynb
# "iiif"   = mei_single_file_iiif_integration.ipynb
# "guided" = mei_pdf_facsimile_guided.ipynb
RESULT = "pdf"

# Port of the preview server in step 5; 0 picks a free one.
PREVIEW_PORT = 0

## 2. Check the result

`RESULTS` says where each notebook leaves its result. If you saved one elsewhere, for example with another `OUTPUT_FOLDER` in the guided notebook, change `folder` here.

The viewer finds the scan of each page through `<pb facs>`, the link from a page break to its `<surface>`. The PDF notebooks write these links; the IIIF notebook does not. This cell adds the missing ones, taking each page from the zone of the bar after the break, as `camat-link-pb-surface --apply` does. Nothing else in the MEI changes.

In [ ]:
RESULTS = {
    "pdf": {
        "notebook": "mei_pdf_facsimile_integration.ipynb",
        "folder": "converted_mei/pdf_tutorial",
        "mei": "mei/Buxtehude-DdT11-app01-suite_pdf_facs_breaks.mei",
        "pages": "pages/pages.json",
        "example": "test_corpus/Buxtehude-DdT11-app01-suite.mei",
        "bar": "10",
    },
    "iiif": {
        "notebook": "mei_single_file_iiif_integration.ipynb",
        "folder": "converted_mei/iiif_tutorial",
        "mei": "bsb00023199_00185_facs_zones.mei",
        "example": "test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei",
        "bar": "5",
    },
    "guided": {
        "notebook": "mei_pdf_facsimile_guided.ipynb",
        "folder": "converted_mei/pdf_guided",
        "mei": "mei/Buxtehude-DdT11-app01-suite_with_facsimile.mei",
        "pages": "pages/pages.json",
        "example": "test_corpus/Buxtehude-DdT11-app01-suite.mei",
        "bar": "10",
    },
}

if RESULT not in RESULTS:
    raise ValueError(f"RESULT must be one of {', '.join(map(repr, RESULTS))}, not {RESULT!r}.")
result = RESULTS[RESULT]
site_folder = resolve_repo_path(result["folder"])
mei_file = site_folder / result["mei"]
needed = [mei_file] + ([site_folder / result["pages"]] if "pages" in result else [])
missing = [path for path in needed if not path.is_file()]
score_only = bool(missing)

if score_only:
    site_folder = resolve_repo_path("converted_mei/web_edition_score_only")
    mei_file = resolve_repo_path(result["example"])
    print(f"✗ No result yet: {display_path(missing[0])} is missing.")
    print(f"  Run {result['notebook']} first, then this notebook again.")
    print(f"  Until then, this notebook builds a site of the score alone, without a facsimile,")
    print(f"  from {display_path(mei_file)} into {display_path(site_folder)}.")
else:
    links = link_pb_to_surface(mei_file)
    for row in links.rows:
        if row.status in ("mismatch", "unresolved"):
            print(f"! {display_path(mei_file)}, line {row.line}: {row.message}")
    if links.updates:
        link_pb_to_surface(mei_file, apply=True)
        print(f"Linked {links.updates} page break{'s' if links.updates != 1 else ''} to their scan page.")
    print(f"✓ Found the result of {result['notebook']}: {display_path(mei_file)}")

## 3. Write the site config

`site.json` describes the site: its title, the works (one MEI file each) and the facsimile. Its paths are relative to its own folder. For the PDF tutorial, the cell writes the guide's worked example. [Site config](../docs/guides/web-edition.md#site-config) lists every key, such as `footer`, `links` and `labels`.

`"images": {"copy": true}` copies the page images into the site, so that the scan appears at once. Copy images only when you may publish them; otherwise leave out `images`, and visitors open their own copy of the PDF.

In [ ]:
if score_only:
    config = {
        "title": "Buxtehude, DdT vol. 11" if RESULT == "iiif" else "Buxtehude, Suite",
        "subtitle": f"The score only: run {result['notebook']} for the facsimile",
        "storage_prefix": "camat-score-only",
        "works": [{"id": "score", "title": "Score", "mei": Path(os.path.relpath(mei_file, site_folder)).as_posix()}],
    }
elif RESULT == "iiif":
    config = {
        "title": "Buxtehude, DdT vol. 11",
        "subtitle": "Image 185 of the Bayerische Staatsbibliothek's scan, as IIIF images",
        "storage_prefix": "buxtehude-iiif-page",
        "works": [{"id": "page185", "title": "Image 185", "mei": result["mei"]}],
        "facsimile": {"type": "iiif"},
    }
else:
    config = {
        "title": "Buxtehude, Suite",
        "subtitle": "DdT vol. 11, appendix no. 1, linked to the BSB scan",
        "storage_prefix": "buxtehude-suite" if RESULT == "pdf" else "buxtehude-suite-guided",
        "works": [{"id": "suite", "title": "Suite", "mei": result["mei"]}],
        "facsimile": {"type": "pdf", "pages": result["pages"], "pdf": True, "images": {"copy": True}},
    }

site_json = site_folder / "site.json"
site_folder.mkdir(parents=True, exist_ok=True)
site_json.write_text(json.dumps(config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Wrote {display_path(site_json)}:")
print(site_json.read_text(encoding="utf-8"))

## 4. Build the site

`build_site` checks the config, the MEI files and the facsimile first and lists every problem together. Only then does it replace `_site/` as a whole. Inside a git repository the output folder must be named `_site`.

The same build from a terminal:

```bash
camat-build-site converted_mei/pdf_tutorial/site.json converted_mei/pdf_tutorial/_site
```

In [ ]:
site_dir = site_folder / "_site"
try:
    report = build_site(site_json, site_dir)
except SiteConfigError as error:
    report = None
    print("✗ The build stopped and wrote nothing:")
    for problem in error.problems:
        print(f"  {problem}")
else:
    works = report["works"]
    print(f"✓ Built {display_path(site_dir)}: {len(works)} work{'s' if len(works) != 1 else ''}, "
          f"{sum(w['measures'] for w in works)} bars ({sum(w['measures_with_zone'] for w in works)} with a zone), "
          f"{len(report['files'])} files, {report['bytes'] / 1024:.0f} KB")
    for work in works:
        print(f"  {work['id']}: {work['facsimile_pages']} facsimile pages, {work['notes']} editorial notes")
        if work["measures_without_id"]:
            print(f"  {work['id']}: {work['measures_without_id']} bars have no xml:id and cannot be selected")

## 5. Preview the site

The cell starts a small web server for `_site/` in the background and prints the link. Only this computer can reach the server. Opening `index.html` directly from disk does not work: browsers load the viewer's modules only over HTTP. The page needs an internet connection, because Verovio loads from jsDelivr.

Run the cell again after a new build; it restarts the server. The same preview from a terminal:

```bash
python -m http.server -d converted_mei/pdf_tutorial/_site
```

and then open <http://localhost:8000/#suite/10>.

**Cloud Jupyter** (Colab, Jupyter4NFDI): the server runs on the cloud machine, so the link does not open in your browser. Download the `_site` folder (zip it first, for example with `shutil.make_archive("site", "zip", site_dir)`) and run the terminal command on your computer.

In [ ]:
class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # no line per request in the notebook


if globals().get("preview") is not None:
    preview.shutdown()
    preview.server_close()
preview = None

if report is None:
    print("Skipped. Build the site in step 4 first.")
else:
    handler = functools.partial(QuietHandler, directory=str(site_dir))
    try:
        preview = http.server.ThreadingHTTPServer(("127.0.0.1", PREVIEW_PORT), handler)
    except OSError as exc:
        print(f"✗ Port {PREVIEW_PORT} cannot be used ({exc.strerror}). Set PREVIEW_PORT = 0 and run step 1 again.")
    else:
        threading.Thread(target=preview.serve_forever, daemon=True).start()
        work_id = config["works"][0]["id"]
        print(f"Serving {display_path(site_dir)}")
        print(f"Open http://127.0.0.1:{preview.server_port}/#{work_id}/{result['bar']}")

## 6. Stop the preview

Run All leaves the server running, so that the link keeps working. When you are done, set `STOP_PREVIEW = True` and run this cell, or restart the kernel.

In [ ]:
STOP_PREVIEW = False

if not STOP_PREVIEW:
    print("The preview keeps running. To stop it, set STOP_PREVIEW = True and run this cell again.")
elif globals().get("preview") is None:
    print("No preview server is running.")
else:
    preview.shutdown()
    preview.server_close()
    preview = None
    print("Stopped the preview server.")

## Publish

Any static web host can serve `_site/` as it is. For GitHub Pages, build into `_site` in a GitHub Actions workflow and deploy that folder; [webern_op27's workflow](https://github.com/egorpol/webern_op27/blob/main/.github/workflows/pages.yml) is an example. See [Publish](../docs/guides/web-edition.md#publish) for what visitors need.

## Your own MEI

1. Link your MEI to its scan with one of the facsimile notebooks, or start from MEI that already has `<zone>`s, `measure/@facs` and page breaks. [What the MEI needs](../docs/guides/web-edition.md#what-the-mei-needs) lists the details.
2. Add your result to `RESULTS` in step 2, or write a `site.json` next to your MEI as in step 3: one entry in `works` per MEI file, and a `facsimile` of type `pdf` or `iiif`. Without `facsimile`, the site shows the score alone.
3. Build and preview it with steps 4 and 5, or from a terminal with `camat-build-site`.

Two finished editions show what a published site can look like: [Webern, Op. 27](https://egorpol.github.io/webern_op27/) (PDF facsimile) and [Buxtehude, DdT vol. 11](https://egorpol.github.io/DdT_1_vol_11/) (IIIF). They came before this template and keep their own viewers.